# lattice-grid-jupyter demo

Edit a pandas DataFrame as an interactive Lattice Grid, with a live round-trip
back to Python. Run this notebook top to bottom.

In [ ]:
import pandas as pd
import numpy as np
from lattice_grid_jupyter import LatticeGridWidget, GRID_VERSION
print("grid version:", GRID_VERSION)

## 1. A DataFrame -> an editable grid

Common dtypes map to grid column types: int/float -> `number`, bool -> `boolean`,
datetime -> `timestamp`, object/string/category -> `text`.

In [ ]:
df = pd.DataFrame({
    "name":   ["Ada", "Grace", "Linus", "Margaret"],
    "score":  [91, 88, 77, 95],
    "ratio":  [0.91, 0.88, 0.77, 0.95],
    "active": [True, False, True, True],
    "joined": pd.to_datetime(["2019-03-01", "2020-07-15", "2018-11-30", "2021-01-05"]),
    "team":   pd.Categorical(["A", "B", "A", "C"]),
})
df

In [ ]:
w = LatticeGridWidget(df)   # localhost notebook => free, unwatermarked
w                           # edit cells here; then run the next cell

## 2. Edits round-trip back to the DataFrame

Edit some cells above, then run this. `w.df` reflects your edits, dtype-preserved.
(For a reproducible demo without a live browser, we replay the same comm payload
the grid emits on an edit.)

In [ ]:
# what a real edit sends over the comm:
w.apply_edit(key="0", col_id="score", value=100)   # Ada 91 -> 100
w.apply_edit(key="1", col_id="active", value=True)  # Grace False -> True
w.df

In [ ]:
# dtypes are preserved
w.df.dtypes

## 3. Missing values, MultiIndex, duplicate index

NaN/NaT/NA -> null. A MultiIndex becomes one read-only column per level. Row
identity is positional, so even duplicate index labels work.

In [ ]:
mi = pd.DataFrame(
    {"q1": [10.0, np.nan, 30.0], "q2": [1, 2, 3]},
    index=pd.MultiIndex.from_tuples([("north","a"),("north","b"),("south","a")],
                                    names=["region","site"]),
)
LatticeGridWidget(mi)

## 4. Python -> grid: live updates

`set_data`, `append_rows` and `delete_rows` push changes into a rendered grid.

In [ ]:
keys = w.append_rows([{"name":"Alan","score":80,"ratio":0.80,"active":True,
                       "joined":pd.Timestamp("2022-02-02"),"team":"B"}])
print("appended row keys:", keys)
w.df.tail(2)

In [ ]:
w.delete_rows(keys)   # remove what we just appended
w.df.tail(2)

## 5. Large DataFrames (100k rows)

Serialized column-major and rendered with virtualization, so it does not freeze.

In [ ]:
n = 100_000
big = pd.DataFrame({
    "id": np.arange(n),
    "val": np.random.rand(n).round(4),
    "flag": np.random.rand(n) > 0.5,
    "label": [f"r{i}" for i in range(n)],
})
import time
t = time.time()
big_w = LatticeGridWidget(big)
print(f"built widget for {n:,} rows in {time.time()-t:.2f}s")
big_w

## 6. Licensing

On `localhost` (the normal notebook case) the grid is free and unwatermarked with
no key. Deployed on a non-localhost origin, pass a key:

```python
w = LatticeGridWidget(df, licence="LG-...")
```

## Offline notebooks

`LatticeGridWidget(df, offline=True)` carries the grid bundle inside the widget,
so it renders with no network at all.